# 공연 종료 후 0~3시간·0~6시간 분석을 위한 KOPIS 필드 점검
현재 저장된 KOPIS 상세에서 공연 시각·러닝타임·기간·시설 정보가 있는지 확인한다. **고정 입력 명세 `jupyter-notebook/20261006/metadata/kopis_analysis_sources.json`이 지정한 스냅샷만 읽는다.** 원본·기존 모형·매출 변수는 변경하지 않고, 네트워크 요청도 하지 않는다.

이 점검의 단위는 **공연 ID별 상세 레코드**다. 요일별 공식 일정 문구, 그 문구를 바탕으로 계산할 수 있는 예정·추정 시각, 실제 날짜별 상연 회차와 실제 시작·종료 시각은 서로 다르다. 확인 결과는 `reports/20261006/analysis_report/time_feasibility/`에 저장한다.

## 1. 고정 스냅샷과 SHA-256 검증
매니페스트 전체 내용이나 인증 설정을 출력하지 않는다. 여기서는 검증된 파일 경로·해시·수집 회차만 사용한다.

In [ ]:
# 현재 구조 및 과거 고정 명세의 경로를 함께 해석합니다.
from pathlib import Path as _ProjectPath
import json as _bootstrap_json
_project_home = next(p for p in [_ProjectPath.cwd(), *_ProjectPath.cwd().parents]
                     if (p / 'environment/project_paths.ipynb').is_file())
_project_notebook = _bootstrap_json.loads((_project_home / 'environment/project_paths.ipynb').read_text(encoding='utf-8'))
for _project_cell in _project_notebook['cells']:
    if 'project-runtime' in _project_cell.get('metadata', {}).get('tags', []):
        exec(compile(''.join(_project_cell['source']), 'environment/project_paths.ipynb', 'exec'), globals())

from pathlib import Path
import hashlib, json, re
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
ROOT = PROJECT_ROOT
OUT = project_path('reports/20261006/analysis_report/time_feasibility')
OUT.mkdir(parents=True, exist_ok=True)
MANIFEST = project_path('jupyter-notebook/20261006/metadata/kopis_analysis_sources.json')
hash_file = lambda path: hashlib.sha256(path.read_bytes()).hexdigest()
manifest_hash = hash_file(MANIFEST)
manifest = project_json_loads(MANIFEST.read_text(encoding='utf-8'))
assert manifest['validated'] is True

In [ ]:
source_records, frames = [], {}
for role in ['performance_details','facility_details']:
    spec = manifest['files'][role]
    path = project_path(spec['path'])
    actual = hash_file(path)
    assert actual == spec['sha256'], f'고정 입력 해시 불일치: {role}'
    frames[role] = pd.read_csv(path, dtype=str, keep_default_na=False)
    source_records.append(dict(role=role,path=spec['path'],expected_sha256=spec['sha256'],
        sha256_before=actual,rows=len(frames[role]),columns=frames[role].columns.tolist()))
performances = frames['performance_details']
facilities = frames['facility_details']
assert performances.mt20id.is_unique and facilities.mt10id.is_unique
N = len(performances)
print(f'공연 상세 {N:,}개 공연 ID / 시설 상세 {len(facilities):,}개 시설 ID / 고정 해시 일치')

**해석:** 이 분모는 수집·검증된 공연 상세 ID 수다. 서울 전체 공연 목록 수나 실제 상연 회차 수, 종로구·마포구의 500m 분석 기여 ID 수와 같지 않다.

## 2. 필드 존재와 빈 값
공연시각 `dtguidance`, 러닝타임 `prfruntime`, 공연기간 `prfpdfrom`·`prfpdto`, 시설 ID `mt10id`·공연장 ID `mt13id`, 시설명 `fcltynm`, 상세 갱신일 `updatedate`를 점검한다. 빈 문자열을 수치 0으로 바꾸지 않는다.

In [ ]:
field_rows = []
fields = ['mt20id','dtguidance','prfruntime','prfpdfrom','prfpdto','mt10id','mt13id','fcltynm','updatedate']
for field in fields:
    exists = field in performances.columns
    count = int(performances[field].str.strip().ne('').sum()) if exists else 0
    field_rows.append(dict(field=field,field_exists=exists,total_rows=N,
        nonempty_rows=count,empty_rows=N-count,nonempty_pct=100*count/N))
field_coverage = pd.DataFrame(field_rows)
display(field_coverage.style.format({'nonempty_pct':'{:.2f}'}))

**해석:** 필드가 채워져 있다는 것은 해당 문구가 저장되었다는 뜻이다. 실제 회차 확인, 과거 일정의 정확성 또는 실제 종료 시각의 관측을 뜻하지 않는다. `updatedate`도 회차 발생 시각이 아니다.

## 3. 시각·러닝타임 문구의 보수적 형식 점검
요일과 `HH:MM` 형식 시각의 존재를 확인하되 날짜별 회차를 생성하지 않는다. 러닝타임은 `1시간 30분`, `50분`처럼 단일 시간 길이인 문구만 분으로 읽는다. `1시간 60분`처럼 시·분 표기가 비정형인 경우는 검토 대상으로 남긴다. 매우 긴 러닝타임도 개별 상연의 실제 지속시간이라고 단정하지 않는다.

In [ ]:
schedule = performances.dtguidance.str.strip()
runtime = performances.prfruntime.str.strip()
clock_pattern = r'(?<!\d)(?:[01]?\d|2[0-3]):[0-5]\d(?!\d)'
clock_tokens = schedule.str.findall(clock_pattern)
has_clock = clock_tokens.str.len().gt(0)
has_weekday = schedule.str.contains(r'[월화수목금토일]요일|HOL',regex=True)
has_explicit_date = schedule.str.contains(r'\d{4}[.\-/년]\s*\d{1,2}|\d{1,2}월\s*\d{1,2}일',regex=True)
date_from = pd.to_datetime(performances.prfpdfrom,format='%Y.%m.%d',errors='coerce')
date_to = pd.to_datetime(performances.prfpdto,format='%Y.%m.%d',errors='coerce')
valid_period = date_from.notna() & date_to.notna() & date_from.le(date_to)

In [ ]:
def parse_runtime(text):
    match = re.fullmatch(r'(?:(\d+)시간)?\s*(?:(\d+)분)?', text)
    if not match or not any(match.groups()):
        return np.nan
    hours, minutes = match.groups()
    if hours is not None and minutes is not None and int(minutes) >= 60:
        return np.nan
    value = 60*int(hours or 0)+int(minutes or 0)
    return float(value) if value > 0 else np.nan
runtime_minutes = runtime.map(parse_runtime)
nominal_ready = has_clock & runtime_minutes.notna() & valid_period & performances.mt10id.str.strip().ne('')
noncanonical_runtime = runtime.ne('') & runtime_minutes.isna()

In [ ]:
metrics = {
    'all_performance_ids':N,
    'schedule_with_clock':int(has_clock.sum()),
    'schedule_with_weekday_or_holiday':int(has_weekday.sum()),
    'schedule_with_holiday_label':int(schedule.str.contains('HOL',regex=False).sum()),
    'schedule_with_explicit_calendar_date_pattern':int(has_explicit_date.sum()),
    'runtime_nonempty':int(runtime.ne('').sum()),
    'runtime_conservatively_parseable':int(runtime_minutes.notna().sum()),
    'runtime_nonempty_needs_review':int(noncanonical_runtime.sum()),
    'runtime_over_6_hours':int(runtime_minutes.gt(360).sum()),
    'valid_period':int(valid_period.sum()),
    'registered_period_is_one_day':int((valid_period & date_from.eq(date_to)).sum()),
    'registered_period_extends_beyond_2025':int((valid_period & date_to.gt('2025-12-31')).sum()),
    'clock_runtime_period_and_facility_available':int(nominal_ready.sum()),
    'official_facility_id_matches_snapshot':int(performances.mt10id.isin(facilities.mt10id).sum())}
structure = pd.DataFrame([{'metric':k,'count':v,'denominator_performance_ids':N,'pct':100*v/N} for k,v in metrics.items()])
display(structure.style.format({'pct':'{:.2f}'}))

**해석:** `clock_runtime_period_and_facility_available`은 예정 종료 시각을 추정하기 위한 최소 문구가 있다는 수다. 완전한 날짜별 일정표를 확정할 수 있다는 수가 아니다. 긴 공연기간에 현재 요일별 일정을 일괄 적용하면 휴연·특별회차·일정 변경·공휴일 중복 처리 오류가 생길 수 있다. 종료가 자정을 넘으면 다음 날짜로 넘어가므로 달력 시각과 시간창 정의도 필요하다.

## 4. 대표 원문 형태와 실제 회차 정보의 유무
다음 표의 문자열은 고정 상세 CSV에 저장된 값이다. 예정 시각에 러닝타임을 더한 계산값을 공식 종료 시각으로 바꾸어 저장하지 않는다. 날짜별 회차 ID·실제 시작·실제 종료·회차 취소 정보가 독립 필드로 존재하는지도 확인한다.

In [ ]:
examples = []
def add_example(label, mask):
    row = performances.loc[mask].sort_values('mt20id').head(1)
    if not row.empty:
        selected = row[['mt20id','prfnm','prfpdfrom','prfpdto','mt10id','prfruntime','dtguidance']].copy()
        selected.insert(0,'example_type',label)
        examples.append(selected)
add_example('단순 요일·시각',schedule.eq('금요일(20:00)') & runtime.eq('1시간 30분'))
add_example('복수 요일·시각 및 공휴일',performances.mt20id.eq('PF132171'))
add_example('장기공연·2025년 이후 종료',performances.mt20id.eq('PF121682'))
add_example('러닝타임 미제공',runtime.eq(''))
add_example('공연시각 미제공',schedule.eq(''))
add_example('러닝타임 비정형 표기',noncanonical_runtime)
example_table = pd.concat(examples,ignore_index=True)
display(example_table)

In [ ]:
explicit_session_columns = [c for c in performances.columns
    if re.search(r'session|episode|round|actual_start|actual_end|cancel|회차|실제시작|실제종료|취소',c,re.I)]
assert explicit_session_columns == []
session_audit = {
    'row_grain':'one_record_per_performance_id',
    'one_record_per_performance_id':bool(performances.mt20id.is_unique),
    'explicit_dated_session_id_column':False,
    'explicit_actual_start_end_column':False,
    'explicit_session_cancellation_column':False,
    'recognized_session_columns':explicit_session_columns,
    'dtguidance_with_explicit_calendar_date_pattern':int(has_explicit_date.sum()),
    'registered_single_day_ids':int((valid_period & date_from.eq(date_to)).sum()),
    'dated_actual_sessions_available_in_this_snapshot':False,
    'note':'단일 날짜의 등록기간도 실제 상연 완료·회차수·실제 종료를 확인한 기록과 다르다.'}
print('현재 상세 스냅샷: 공연 ID당 1행, 날짜별 실제 회차·실제 종료·회차 취소 필드 없음')

**해석:** 예를 들어 공식 일정 문구가 19:30 시작이고 공식 러닝타임이 90분이면 종료 21:00을 **계산으로 추정**할 수 있다. 그러면 종료 후 0~3시간 창은 21:00~다음날 00:00, 0~6시간 창은 21:00~다음날 03:00이라는 예정 기반 창이 된다. 실제 지연, 인터미션 처리, 조기 종료, 취소 여부를 확인한 실제 종료 시각은 아니다.

이 점검만으로 정확한 공연 후 매출을 얻을 수는 없다. 그 분석에는 해당 날짜와 시각의 매출 관측이 별도로 필요하다. 분기×고정 시간대별 매출 합계는 공연이 있었던 날짜와 종료 직후 창에 대한 매출을 따로 식별하지 못한다. 본 노트북은 매출자료를 새로 재구성하거나 기존 21~24시 결과를 변경하지 않는다.

## 5. 결과와 입력 불변성 저장
대표 형식과 필드 제공률, 구조 요약 CSV 3개와 활용가능성·해시 JSON 1개를 저장한다. 원문 스냅샷의 실행 전후 해시를 비교한다.

In [ ]:
tables = {'01_field_coverage.csv':field_coverage,
          '02_timing_structure.csv':structure,
          '03_schedule_examples.csv':example_table}
for name,table in tables.items():
    table.to_csv(OUT/name,index=False,encoding='utf-8-sig')
for record in source_records:
    record['sha256_after'] = hash_file(project_path(record['path']))
    record['unchanged'] = record['sha256_before'] == record['sha256_after']
assert all(record['unchanged'] for record in source_records)
assert hash_file(MANIFEST) == manifest_hash

In [ ]:
summary = {
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'run_id':manifest['run_id'],'manifest_path':MANIFEST.relative_to(ROOT).as_posix(),
    'manifest_sha256_before':manifest_hash,'manifest_sha256_after':hash_file(MANIFEST),
    'source_files':source_records,'performance_id_denominator':N,
    'scope_note':'검증된 공연 상세 ID 표본이며 서울 전체 목록·실제 상연회차·500m 분석 ID 분모와 다름',
    'field_availability':field_coverage.to_dict('records'),'timing_metrics':metrics,
    'session_audit':session_audit,
    'nominal_end_time_computable_for_some_records':True,
    'computed_end_time_is_official_observed_end_time':False,
    'historical_schedule_verified':False,
    'exact_post_performance_sales_window_available_from_this_snapshot':False,
    'feasible_use':'공식 요일·예정시각과 러닝타임에서 예정 종료 시각을 가정하여 보조 노출지표를 만드는 준비자료',
    'required_for_exact_analysis':['검증된 날짜별 실제 회차·시작·종료·취소 정보',
        '같은 날짜·시각 단위의 매출자료','야간 자정 경과 및 여러 공연 시간창 중복 처리 규칙'],
    'limitations':['요일·시각 안내는 실제 회차 로그가 아니다.',
        '공연기간은 일자별 모든 회차·휴연·특별일정을 확정하지 못한다.',
        '2026 수집 상세의 일정이 2023~2025 과거 전기간에 동일했다고 보장할 수 없다.',
        '러닝타임으로 계산한 종료시각은 실제 지연·인터미션·취소 등을 확인하지 못한 추정값이다.',
        '실제 종료 후 매출은 날짜·시각별 매출이 필요하며 분기 고정 시간대 합계만으로 복원할 수 없다.'],
    'all_inputs_unchanged':True,'network_requests':0,'new_model_fits':0,
    'output_sha256':{name:hash_file(OUT/name) for name in tables}}
(OUT/'feasibility_summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
print(f'저장 완료: CSV {len(tables)}개 + JSON 1개; 원본·매니페스트 해시 불변')

**판단:** 공연시각·러닝타임 문구는 상당수 확보되어 예정 종료 시각에 따른 보조 분석을 검토할 수 있다. 그러나 저장된 공연 상세만으로 실제 회차 종료 시점을 관측했다고 할 수 없다. 날짜·시각별 매출도 함께 확보해야 정확한 ‘공연 종료 후 0~3시간·0~6시간 매출’을 직접 분석할 수 있다.